1. Import Libraries

In [2]:
## kernel - base
"""
Persist Salesforce object field metadata (df_meta) into a SQL Server table.

- Creates the table if it doesn't exist
- Appends new rows OR performs an UPSERT (MERGE) if enabled
- Uses fast_executemany for performance
"""

#Importing Other Libraries
from simple_salesforce import Salesforce,SalesforceLogin,SalesforceAuthenticationFailed,SFType
import pandas as pd
import json
import numpy as np
import time as t
import uuid
import os
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from collections import OrderedDict

from core_functions import *
from sf_source_objects import *
from sf_target_objects import *

In [3]:
# ----------------------------
# PARAMETERS
# 1) CONFIGURATION
# ----------------------------
p_db_server  = 'Rhombus-SQL-Ser'
# p_db  = 'ADVICE_ETL_PRD'
p_db  = 'SalesforceMigration'
p_sch = 'ids'
p_tbl_fd = 'sf_object_fd'
p_tbl_pl = 'sf_object_pl'
p_sf_src = 'hubprod'
# p_sf_tgt = 'rhpartial'
# p_sf_tgt = 'rhdata'
# p_sf_tgt = 'rhuat'
# p_sf_tgt = 'datasb'
# p_sf_tgt = 'flow2sb'
p_sf_tgt = 'rhsit'

# ----------------------------
# CONSTANTS
# WRITE: APPEND or UPSERT
# ----------------------------
UPSERT_ENABLED = True  # set False to just append/replace
WRITE_CHUNK_SIZE = 10_000

connection_string = (
    f"mssql+pyodbc://@{p_db_server}/{p_db}"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&Trusted_Connection=yes"
    "&fast_executemany=True"
)


## Configure Logging
This section initializes a logging framework to monitor the ETL workflow’s execution and facilitate debugging by:
Creating an isolated logger named ETL_Salesforce.
Setting the log level to INFO to capture all important messages.
Outputting formatted messages (with timestamps and levels) to the console.
Disabling propagation to prevent duplicate log entrie

In [4]:
# Configuring logging
import logging
 
# Create a dedicated logger for the ETL process
logger = logging.getLogger('ETL_Salesforce')
logger.setLevel(logging.INFO)
 
# Clear any existing handlers to avoid duplicates
logger.handlers = []  # Remove all handlers from this logger
 
# Create a new StreamHandler with the desired format
handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter("%(asctime)s [%(levelname)s] %(message)s"))
logger.addHandler(handler)
 
# Disable propagation to prevent root logger interference
logger.propagate = False

## Establish Connections
This section establishes a connection to the source Salesforce environment using credentials securely fetched from Google Colab’s secrets storage.
Connecting to the Source Salesforce Instance:

In [5]:
#Connecting To The Source Salesforce Instance
from secrets_ids import secrets
url = secrets.get(p_sf_src, {}).get('site', 'Unknown')
username = secrets.get(p_sf_src, {}).get('username', 'Unknown')
pswd = secrets.get(p_sf_src, {}).get('password', 'Unknown')
token = secrets.get(p_sf_src, {}).get('token', 'Unknown')

src_sf_instance = connect_to_salesforce(url=url, username=username, password=pswd, token=token, domain='login', logger=logger)
 
logger.info(f"Source SF Instance Session ID = {src_sf_instance.session_id}")

2026-09-03 21:03:19,431 [INFO] Connected successfully to https://ioofadviserhub.my.salesforce.com
2026-09-03 21:03:19,432 [INFO] Source SF Instance Session ID = 00D6F0000029Bzd!AQEAQLCqrN.2fpqDSy9p88V47KK4vvNDqzb4Z3sXh.mUK3Gp77liZ0PlkDConuXILZbkEKsReMheJmASmehfJSL0xcFQuqup


To maintain security, credentials are not hardcoded. Instead, they are stored in Google Colab’s Secrets vault. The application dynamically accesses these stored values (src_username, src_password, and src_token) via the userdata.get('secret_name') method. The fetched credentials are passed to the connect_to_salesforce function to authenticate and create a Salesforce client instance.

Connecting to the Destination Salesforce Instance:

In [5]:
#Connecting To The Destination Salesforce Instance
url_tgt = secrets.get(p_sf_tgt, {}).get('site', 'Unknown')
username_tgt = secrets.get(p_sf_tgt, {}).get('username', 'Unknown')
pswd_tgt = secrets.get(p_sf_tgt, {}).get('password', 'Unknown')
token_tgt = secrets.get(p_sf_tgt, {}).get('token', 'Unknown')

dst_sf_instance = connect_to_salesforce(url=url_tgt, username=username_tgt, password=pswd_tgt, token=token_tgt, domain='test', logger=logger)
 
logger.info(f"Destination SF Instance Session ID = {dst_sf_instance.session_id}")

2026-09-03 11:10:45,091 [INFO] Connected successfully to https://test.salesforce.com
2026-09-03 11:10:45,092 [INFO] Destination SF Instance Session ID = 00DBm00000CWPlx!AQEAQJisfayl5n7.FwzSKP.ctprm4b84oZySLRyBaIa8MAzE8jvHcOxYPPc05qLr3sPwjfiGuz76lF5JCcIqb2.XZT8KKTbS


## Data Extraction Phase
This section handles the data extraction process by:
Defining the Salesforce object to query (Account) and the key reference column (Name) used for data validation and uniqueness checks.
Constructing a SOQL query string to retrieve the Id, Name, and Phone fields while limiting the results to three records to get a small sample of data for testing.
Calling the extract_query function with the Salesforce connection and SOQL query; This function executes the query and loads the results into a panda DataFrame (object_df).
Logging the count of non-null values per column to identify missing data.
Logging the total number of records extracted.
Logging the count of unique values in the reference column (Name).
Checking for duplicates in the reference column and logging a warning if any are found.
Displaying the first five rows of the extracted DataFrame to give a preview of the data structure and content.

In [4]:
# LIST OF OBJECTS TO EXTRACT from Source SF Instance
# from sf_toTransfer_objects import TOTRANSFER_OBJECTS
from sf_source_objects import SOURCE_OBJECTS

lst_objects_copy = SOURCE_OBJECTS

In [11]:
lst_objects_copy = [
    # 'A1_Support_Staff_Registrations__c',
    # 'Account',
    # 'AccountContactRelation',
    # 'Accreditations__c',
    # # 'ACR_Case__c',
    # 'Adviser_Academy__c',
    # 'Adviser_ASN_Snapshot__c',
    # 'AdviserHUB_Wave_Definition__c',
    # 'AdviserOne_Event__c',
    # 'AdviserOne_Site_Content__c',
    # 'ANZ_Lookback__c',
    'Approval__c',
    # 'Approval__c_hd',
    # 'Approved_Non_standard_Template__c',
    'Area_of_Advice__c',
    # 'articleBody__Article_Body_Setting__mdt',
    # 'ASF_CSA__c',
    # 'ASF_CSA_Case__c',
    # 'ASF_CSA_Case_dv__c',
    # 'ASF_CSA_New__c',
    'ASN_Indicator_History__c',
    'Assurance_Event__c',
    'Assurance_Review__c',
    # 'Attachment',
    # 'Australian_Public_Holiday__c',
    # 'AuthSession',
    # 'bar__Backup_Object__c',
    # 'bar__Backup_Policy__c',
    # 'bar__Log__c',
    'BID_Case__c',
    # 'BrandingSet',
    # 'BrandingSetProperty',
    'BrandTemplate',
    'Breach_Reporting__c',
    # 'bridges_rpt_referral2__c',
    # 'bridges_rpt_referral__c',
    # 'bridges_rpt_referral_revenue2__c',
    # 'bridges_rpt_referral_revenue__c',
    # 'bridges_rpt_refunds__c',
    # 'bridges_tpo_diary__c',
    # 'bridges_tpo_entity__c',
    # 'bridges_tpo_entity_dv__c',
    # 'bridges_tpo_master_member__c',
    # 'bridges_tpo_master_member_dv__c',
    # 'bridges_tpo_member_securities__c',
    # 'bridges_tpo_member_securities_dv__c',
    # 'bridges_tpo_thread_task__c',
    # 'bridges_tpo_thread_task_bfpe__c',
    # 'bridges_tpo_thread_task_bfpe_dv__c',
    # 'bridges_tpo_thread_task_dv__c',
    # 'bridges_tpo_user__c',
    # 'bridges_tpo_user_dv__c',
    'BusinessHours',
    # 'BusinessProcess',
    'Calculation__c',
    'Campaign',
    # 'CampaignInfluenceModel',
    'CampaignMember',
    # 'CampaignMemberStatus',
    'Case',
    'Case_Snapshot__c',
    'CaseComment',
    # 'CaseStatus',
    'Client_File__c',
    'Client_Service_Model_Implementation__c',
    # 'ClientBrowser',
    # 'cloudx_cms__SS_Carousel__c',
    'CommPay_Supplier__c',
    # 'Community',
    'Complained_Advisers__c',
    'Consequence_Management_Forum_Entry__c',
    'Contact',
    # 'ContentDistribution',
    # 'ContentDistributionView',
    # 'ContentDocument',
    # 'ContentFolder',
    # 'ContentFolderLink',
    # 'ContentVersion',
    # 'ContentWorkspace',
    # 'ContentWorkspaceDoc',
    # 'ContentWorkspaceMember',
    # 'ContentWorkspacePermission',
    # 'ContractStatus',
    # 'CSA_Case__c',
    # 'CSA_Case_dv__c',
    'CSA_Fee_Switch_Off_History__c',
    'CSO__c',
    # 'CSOAdviser__c',
    # 'CustomBrand',
    # 'CustomBrandAsset',
    # 'Dashboard',
    # 'DashboardComponent',
    # 'DataType',
    # 'Document',
    # 'DuplicateRecordItem',
    # 'DuplicateRecordSet',
    'Email_Log__c',
    'EmailMessage',
    'EmailTemplate',
    # 'EntityDefinition',
    'EnhancedLetterhead',
    'Event',
    'EventRelation',
    # 'EventWhoRelation',
    # 'FiscalYearSettings',
    'Folder',
    'Group',
    'GroupMember',
    # 'hmq__Headlines__c',
    # 'Holiday',
    'Incident_File_Review__c',
    'InitiativeChecklist__c',
    # 'Item__c',
    # 'Knowledge__ka',
    # 'Knowledge__kav',
    # 'Knowledge__ViewStat',
    # 'Knowledge__VoteStat',
    # 'KnowledgeArticle',
    # 'KnowledgeArticleVersion',
    # 'KnowledgeArticleViewStat',
    # 'KnowledgeArticleVoteStat',
    # 'Lead',
    # 'LeadStatus',
    'Library_Text__c',
    # 'ListView',
    # 'ListViewChart',
    # 'LoginIp',
    'LookupName__c',
    'Master_Item__c',
    'Master_module__c',
    'Master_Questions__c',
    'Master_Topic__c',
    'Master_workshop__c',
    'MasterQuestions_Targetedreviews__c',
    'Module__c',
    'NAPW__c',
    # 'OauthToken',
    'Opportunity',
    # 'OpportunityStage',
    'Organization',
    'Paraplanning__c',
    'PermissionSet',
    'PermissionSetAssignment',
    'PermissionSetGroup',
    'PermissionSetLicense',
    # 'PartnerRole',
    # 'Period',
    'Pricebook2',
    # 'ProcessDefinition',
    # 'ProcessInstance',
    # 'ProcessInstanceNode',
    # 'ProcessInstanceStep',
    # 'ProcessInstanceWorkitem',
    # 'ProcessNode',
    'Product2',
    'Product_Questions__c',
    'Profile',
    'PSA__c',
    # 'Publisher',
    'QA__c',
    'QA_Assignment_Hisotry__c',
    # 'QDC_Training__c',
    'Qualification__c',
    'Qualification_Type__c',
    'Question_Group__c',
    'Question_Responses__c',
    # 'QueueSobject',
    # 'QuickText',
    # 'RecentlyViewed',
    'RecordType',
    'Refund_Request__c',
    'Registered_Training_Organisation__c',
    # 'Report',
    'Response_Finding__c',
    # 'Reviewer_Accreditation__c',
    'Reviews_Accounted__c',
    # 'SearchActivity',
    'SecondaryPracticeAdviserHistory__c',
    'SelectedQuestions_Targetedreviews__c',
    'Snapshot_Data__c',
    # 'SolutionStatus',
    # 'sortablegrid__SDG__c',
    # 'sortablegrid__SDG_Action__c',
    # 'sortablegrid__SDG_Field__c',
    # 'sortablegrid__sdg_Preferences__c',
    'Special_Conditions__c',
    # 'StaticResource',
    'Target__c',
    'Task',
    # 'TaskPriority',
    'TaskRelation',
    # 'TaskStatus',
    # 'TaskWhoRelation',
    'Third_Party_Provider__c',
    # 'Topic',
    'Topic__c',
    'Tracking__c',
    # 'UndecidedEventRelation',
    'User',
    # 'UserAppInfo',
    'UserLicense',
    # 'UserPermissionAccess',
    'UserRole',
    'Vetting_Result__c',
    # 'Wealth_Advice_Referral_Program__c',
    'Wealth_Central_Adoption__c',
    # 'WorkAccess',
    # 'WorkBadgeDefinition',
    'Workshop__c',
]


### NOTE - TRUNCATE TABLE for the object before re-populating from SF

In [16]:
# extract data for each object from source SF instance and load into SQL Server
# to load one object, include only the object in the list below
lst_objects_copy = [
    # 'CSA_Case__c'
    # 'ASF_CSA_Case_dv__c'
    # 'CSA_Case_dv__c'
    # 'CSA_Fee_Switch_Off_History__c'
    # 'ASN_Indicator_History__c'
    'Attachment'
    # 'Assurance_Review__c'
    # 'RecordType'
    # 'Contact'
    # 'User'
    # ,'Profile'
    # ,'UserRole'
    # ,'Account'
    # 'Calculation__c'
    # 'PermissionSetAssignment'
    # ,'PermissionSet'
    # ,'PermissionSetGroup'
    # ,'PermissionSetLicense'
    # ,'UserLicense'
]

for sfo in lst_objects_copy:
    print(f"Processing {sfo}.")
    q_sql_select = f"""
    WITH cols AS (
        SELECT
            [object_api_name],
            [column_list] = STRING_AGG(convert(varchar(max),[field_api_name]),', ')
                WITHIN GROUP (ORDER BY [object_api_name])
        FROM [SalesforceMigration].[ids].[sf_object_meta] (nolock)
        where 1=1
        and [object_api_name] = '{sfo}'
        GROUP BY [object_api_name]
    )
    SELECT
        'SELECT '
            + [column_list] + CHAR(13) + CHAR(10) 
            + ' FROM ' + [object_api_name] 
    FROM cols
    ORDER BY [object_api_name]
    """

    # Merge into table
    engine = get_engine(connection_string)
    with engine.begin() as conn:
        rst_select = conn.execute(text(q_sql_select))
        # result = connection.execute(text(f"SELECT top 10 * FROM [{p_sch}].[{tgt_tbl}]"))
        # columns = rst_select.keys()
        rows = rst_select.fetchall()
        soql_select = rows[0][0]
        ## enable the line below to filter by Licensee__c = 'Shadforth' (or adjust as needed) - useful for extracting large objects like Question_Responses__c
        # soql_select = soql_select + " WHERE Licensee__c = 'Shadforth' and CreatedDate > 2026-01-01T00:52:58.136+10:00 "
        # soql_select = soql_select + " and Is_Active__c = 'Yes' " 
        # soql_select = soql_select + " and Commencement_Date__c = NULL and Adviser_Name__c > 'Z' " 
        # soql_select = soql_select + " and Is_Active__c = 'No' and Commencement_Date__c >= 1970-07-01 and Commencement_Date__c < 2025-01-01 " 
        ## END
        print(soql_select)
        # ls_target_tbl = df_target_tbl.columns.tolist()
        conn.commit()
        # conn.close()

    df_object = extract_query(src_sf_instance,soql_select,logger=logger)

    df_object_normalized = normalize_df(df_object)

    engine = get_engine(connection_string)

    with engine.begin() as conn:
        # write_upsert(engine, df_final, q_sql_createTable_fd, temp_fd, q_sql_createTemp_fd, q_sql_merge_fd)
        df_object_normalized.to_sql(
            schema='sf',
            name=sfo,
            con=engine,  # use raw connection for temp table visibility
            if_exists="append",
            index=False
        )


Processing Attachment.
2026-09-04 02:27:23,777 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-04 02:27:23,779 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-04 02:27:23,783 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-04 02:27:23,784 INFO sqlalchemy.engine.Engine [generated in 0.00115s] ()
2026-09-04 02:27:23,786 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-04 02:27:23,787 INFO sqlalchemy.engine.Engine [generated in 0.00081s] ()
2026-09-04 02:27:23,789 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-04 02:27:23,790 INFO sqlalchemy.engine.Engine [generated in 0.00093s] ()
2026-09-04 02:27:23,794 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-04 02:27:23,795 INFO sqlalchemy.engine.Engine 
    WITH cols AS (
        SELECT
            [object_api_name],
            [column_list] = STRI

2026-09-04 02:27:23,801 [INFO] Executing SOQL query: SELECT Id, IsDeleted, ParentId, Name, IsPrivate, ContentType, BodyLength, Body, OwnerId, CreatedDate, CreatedById, LastModifiedDate, LastModifiedById, SystemModstamp, Description, IsEncrypted
 FROM Attachment
2026-09-04 02:27:24,351 [INFO] Fetching next batch: /services/data/v59.0/query/0r8xx5qZwt5qOiQAEU-2000
2026-09-04 02:27:24,654 [INFO] Fetching next batch: /services/data/v59.0/query/0r8xx5qZwt5qOiQAEU-4000
2026-09-04 02:27:24,974 [INFO] Fetching next batch: /services/data/v59.0/query/0r8xx5qZwt5qOiQAEU-6000
2026-09-04 02:27:25,301 [INFO] Fetching next batch: /services/data/v59.0/query/0r8xx5qZwt5qOiQAEU-8000
2026-09-04 02:27:25,628 [INFO] Fetching next batch: /services/data/v59.0/query/0r8xx5qZwt5qOiQAEU-10000
2026-09-04 02:27:25,925 [INFO] Fetching next batch: /services/data/v59.0/query/0r8xx5qZwt5qOiQAEU-12000
2026-09-04 02:27:26,220 [INFO] Fetching next batch: /services/data/v59.0/query/0r8xx5qZwt5qOiQAEU-14000
2026-09-04 02:

2026-09-04 02:27:42,536 INFO sqlalchemy.engine.Engine SELECT CAST(SERVERPROPERTY('ProductVersion') AS VARCHAR)
2026-09-04 02:27:42,537 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-09-04 02:27:42,539 INFO sqlalchemy.engine.Engine SELECT schema_name()
2026-09-04 02:27:42,540 INFO sqlalchemy.engine.Engine [generated in 0.00093s] ()
2026-09-04 02:27:42,543 INFO sqlalchemy.engine.Engine SELECT CAST('test max support' AS NVARCHAR(max))
2026-09-04 02:27:42,544 INFO sqlalchemy.engine.Engine [generated in 0.00093s] ()
2026-09-04 02:27:42,546 INFO sqlalchemy.engine.Engine SELECT 1 FROM fn_listextendedproperty(default, default, default, default, default, default, default)
2026-09-04 02:27:42,546 INFO sqlalchemy.engine.Engine [generated in 0.00083s] ()
2026-09-04 02:27:42,548 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-04 02:27:42,565 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-09-04 02:27:42,660 INFO sqlalchemy.engine.Engine SELECT [INFORMATION_SCHEMA].[TABLES].[TABLE_NAME] 